# DB-QSP
https://arxiv.org/pdf/2504.01077

In [1]:
from qrisp import *
from qrisp.operators import X, Y, Z
from qrisp.jasp import q_fori_loop, q_cond, check_for_tracing_mode
from jax import lax
import scipy as sp
import numpy as np
import jax.numpy as jnp
from copy import deepcopy

## Example model: XXZ

In [2]:
from qrisp.vqe.problems.heisenberg import create_heisenberg_init_function, heisenberg_problem, create_heisenberg_hamiltonian
L = 5
G = nx.Graph()
G.add_edges_from([(k,(k+1)%L) for k in range(L-1)]) 
J = 1
B = 0.5
H = create_heisenberg_hamiltonian(G, J, B)
print(H)

X(0)*X(1) + X(1)*X(2) + X(2)*X(3) + X(3)*X(4) + Y(0)*Y(1) + Y(1)*Y(2) + Y(2)*Y(3) + Y(3)*Y(4) + 0.5*Z(0) + Z(0)*Z(1) + 0.5*Z(1) + Z(1)*Z(2) + 0.5*Z(2) + Z(2)*Z(3) + 0.5*Z(3) + Z(3)*Z(4) + 0.5*Z(4)


In [3]:
# Define scaling factor
F = 1

def exp_H(qv, t):
    H.trotterization(method='commuting')(qv,t/F,5)

# Hamiltonian simulation via second order Suzuki-Trotter formula with 2 steps
def exp_H_2(qv, t):
    H.trotterization(order=2,method='commuting')(qv,t/F,2)

# Calculate E and V

In [4]:
# in qrisp
def calculate_EV(H, state_prep):
    H_2 = H**2
    E = H.expectation_value(state_prep, diagonalisation_method="commuting")()
    E_2 = H_2.expectation_value(state_prep, diagonalisation_method="commuting")()
    
    V = E_2 - E**2
    
    return E, V

# matrix, for tests only
def compute_moments(psi, H):
    psi = np.array([psi]).transpose()
    E = (psi.conj().T @ H @ psi)[0,0].real
    S = (psi.conj().T @ H @ H @ psi)[0,0].real
    return E, S, S - E**2

## calculate s and phase
$$
\frac{(H-zI)\ket{\Psi}}{\|(H-zI)\ket\Psi\|}   =e^{i\theta\Psi}e^{s_{\Psi}[\Psi, H]}\ket\Psi.
$$

with $s_k = \frac{-1}{\sqrt{V_k}}\arccos\left(\frac{|E_{k}-z_{k}|}{\sqrt{V_{k}+|E_{k}-z_{k}|^2}}\right)$
and $\theta_k = \arg\left(\frac{E_k-z_k}{|E_k-z_k|}\right).$

In [5]:
def QSP_unitary_synthesis_params(E, V, z):
    diff = E - z
    s = -1/jnp.sqrt(V)*jnp.arccos(jnp.abs(diff)/jnp.sqrt(V+jnp.abs(diff)**2))
    theta = jnp.angle(diff)
    return s, theta

## DB-QSP steps
$$
\frac{(H-zI)\ket{\Psi}}{\|(H-zI)\ket\Psi\|}   =e^{i\theta\Psi}e^{s_{\Psi}[\Psi, H]}\ket\Psi.
$$
$$
e^{s_\Psi[\Psi,H]} = \left(
e^{is_\Psi^{(N)} \Psi}e^{is_\Psi^{(N)} H}
e^{-is_\Psi^{(N)} \Psi}e^{-is_\Psi^{(N)} H}
\right)^N \nonumber+O(s_\Psi^{3/2}/\sqrt N)\ , 
$$


### Numerical checks of DB-QSP

In [6]:
# Group commutator formula
psi = np.zeros(2**L)
psi[2**(L-2)] = 1
psi = psi/np.linalg.norm(psi)
H_matrix = H.to_array()
psi_dm = np.outer(psi, psi.conj())
s = -0.2

comm = psi_dm@H_matrix - H_matrix@psi_dm
U_exact = sp.linalg.expm(s*comm)
# build the GC approx once
N = 5
for i in range(1, N+1):
    a = np.sqrt(abs(s/i))
    P, Hm = psi_dm, H_matrix
    A, B = 1j*a*P, 1j*a*Hm
    U_gc = np.eye(2**L, 2**L)
    for _ in range(i):
        U_gc = sp.linalg.expm(A) @ sp.linalg.expm(B) @ sp.linalg.expm(-A) @ sp.linalg.expm(-B) @ U_gc

    # Compare them:

    print(f"N={i} ‖U_exact - U_gc‖₂ =", np.linalg.norm(U_exact - U_gc))

N=1 ‖U_exact - U_gc‖₂ = 0.7996376392834138
N=2 ‖U_exact - U_gc‖₂ = 0.6324286670868754
N=3 ‖U_exact - U_gc‖₂ = 0.5365690210112508
N=4 ‖U_exact - U_gc‖₂ = 0.47382639100210594
N=5 ‖U_exact - U_gc‖₂ = 0.4288378406008772


In [7]:
def qsp_gc_expect_1s(psi, H_matrix, z, N=5):
    psi = np.array([psi]).flatten()
    psi_dm = np.outer(psi, psi.conj())
    s, theta = QSP_unitary_synthesis_params(*compute_moments(psi, H_matrix)[::2], z)
    s_ = np.sqrt(np.abs(s/N))
    U_qsp_gc = np.eye(H_matrix.shape[0], dtype=complex)
    for _ in range(N):
        U_qsp_gc = (
            sp.linalg.expm(1j*theta*psi_dm)
            @ sp.linalg.expm(1j*s_*psi_dm)
            @ sp.linalg.expm(1j*s_*H_matrix)
            @ sp.linalg.expm(-1j*s_*psi_dm)
            @ sp.linalg.expm(-1j*s_*H_matrix)
            @ U_qsp_gc
        )
    psi_final = U_qsp_gc @ psi
    E = np.vdot(psi_final, H_matrix @ psi_final).real
    V = np.vdot(psi_final, H_matrix @ H_matrix @ psi_final).real - E**2
    return E, V, psi_final

def qsp_gc_expect(psi, H_matrix, z, N=5):
    # z is list
    for zk in z:
        E, V, psi = qsp_gc_expect_1s(psi, H_matrix, zk, N)
    return E, V, psi

In [8]:
# example numpy calculation
psi = np.zeros(2**L)
psi[2**(L-2)] = 1
psi = psi/np.linalg.norm(psi)
H_matrix = H.to_array()
psi_dm = np.outer(psi, psi.conj())

zk = -0.2

# target state
I = np.eye(2**L, 2**L)
psi_target = (H_matrix-zk*I)@ psi
psi_target /= np.linalg.norm(psi_target)

# db-qsp state
psi_qsp = (H_matrix - zk * I)@psi
psi_qsp /= np.linalg.norm(psi_qsp)
E_qsp_gc, V_qsp_gc, psi_qsp_gc = qsp_gc_expect(psi, H_matrix, [zk], N=5)
print("     Fidelity", abs(np.vdot(psi_target, psi_qsp))**2)
print("     Fidelity_GC", abs(np.vdot(psi_target, psi_qsp_gc))**2)
E_qsp = np.vdot(psi_qsp, H_matrix @ psi_qsp).real
V_qsp = np.vdot(psi_qsp, H_matrix @ H_matrix @ psi_qsp).real - E_qsp**2
print("After DB-QSP", (E_qsp, V_qsp))
print("After DB-QSP_GC", (E_qsp_gc, V_qsp_gc))

     Fidelity 0.9999999999999991
     Fidelity_GC 0.7226945770143561
After DB-QSP (np.float64(4.73232323232323), np.float64(2.988266503418032))
After DB-QSP_GC (np.float64(3.8385758654375617), np.float64(6.538545485626658))


### 1 step

In [ ]:
# Apply k steps of DB-QSP (recursively)
def DB_QSP(qarg, U0, exp_H, N, s, theta, k=1):
    # s, theta are lists
    
    if k == 0:
        # |Psi_0> = U0|0>
        U0(qarg)
        
    else:

        def conjugator(qarg):
            with invert():
                DB_QSP(qarg, U0, exp_H, N, s, theta, k-1)
            
        def reflection(qarg, t_):
            with conjugate(conjugator)(qarg):
                if isinstance(qarg,QuantumArray):
                    qubits = sum([qv.reg for qv in qarg.flatten()], [])
                    mcp(t_, qubits, ctrl_state=0, method="khattar")
                else:
                    mcp(t_, qarg, ctrl_state=0, method="khattar")

        DB_QSP(qarg, U0, exp_H, N, s, theta, k-1)

        s_ = jnp.sqrt(jnp.abs(s[k-1])/N)
        theta_ = theta[k-1]

        for _ in range(N):
            exp_H(qarg, s_)
            reflection(qarg, -s_)
            exp_H(qarg, -s_)
            reflection(qarg, s_)

        reflection(qarg, theta_)

        return qarg

In [19]:
def U0(qv):
    x(qv[1])

def state_prep():
    qarg = QuantumVariable(L)
    U0(qarg) # Prepares the initial state |psi>
    return qarg

z = -0.2
N = 5
E, V = calculate_EV(H, state_prep)
s, theta = QSP_unitary_synthesis_params(E, V, z)
print("     s, theta", (s, theta))
print("Initial EV", E, V)

E1, V1 = calculate_EV(H, lambda: DB_QSP(QuantumVariable(L), U0, exp_H_2, N, [s], [theta]))
print("We expect:", E_qsp_gc, V_qsp_gc)
print("DB-QSP circuit:", E1, V1)

     s, theta (Array(-0.36428455, dtype=float64, weak_type=True), Array(0., dtype=float64))
Initial EV 1.4947962814508529 8.016618327933655
We expect: 3.8385758654375617 6.538545485626658                                      
DB-QSP circuit: 3.8537045774169325 6.4689948588796256


### 2 steps

In [11]:
z_list = [-0.2, 0.1j]
E_qsp_gc2, V_qsp_gc2, psi_qsp_gc2 = qsp_gc_expect(psi, H_matrix, z_list, N=5)
print("After DB-QSP with 2 steps", (E_qsp_gc2, V_qsp_gc2))

After DB-QSP with 2 steps (np.float64(4.650152654263886), np.float64(3.7104331188430173))


In [ ]:
# 2nd step
z_list = [-0.2, 0.1j]
E_qsp_gc2, V_qsp_gc2, psi_qsp_gc2 = qsp_gc_expect(psi, H_matrix, z_list, N=5)
print("After DB-QSP with 2 steps", (E_qsp_gc2, V_qsp_gc2))
s1, theta1 = QSP_unitary_synthesis_params(E1, V1, z_list[1])
def U1(qv):
    DB_QSP(qv, U0, exp_H_2, N, s, theta)

E2, V2 = calculate_EV(H, lambda: DB_QSP(QuantumVariable(L), U1, exp_H_2, N, s1, theta1))
print("We expect:", E_qsp_gc2, V_qsp_gc2)
print("DB-QSP circuit:", E2, V2)

After DB-QSP with 2 steps (np.float64(4.650152654263886), np.float64(3.7104331188430173))
We expect: 4.650152654263886 3.7104331188430173                                      
DB-QSP circuit: 4.586064057771141 4.033134637576765


## multiple steps

$$
\frac{p(H)\ket{\Psi_0}}{\|p(H)\ket{\Psi_0}\|}=\prod_{k=0}^{K-1} e^{i \theta_k \Psi_{k}}  e^{s_{k}[\Psi_{k},H]}\ket{\Psi_0},
$$

In [20]:
# Apply multiple steps of DB-QSP 
def DB_QSP_outer(U0, H, exp_H, z_list, N=2):

    L = H.find_minimal_qubit_amount()
    s_list = []
    theta_list = []

    def state_prep():
        qv = QuantumVariable(L)
        U0(qv)
        return qv

    E, V = calculate_EV(H, state_prep)
    print("Initial EV", E, V)
    E_list = [E]
    V_list = [V]
    
    for k, zk in enumerate(z_list):
        s, theta = QSP_unitary_synthesis_params(E, V, zk)
        s_list.append(s)
        theta_list.append(theta)

        E, V = calculate_EV(H, lambda: DB_QSP(QuantumVariable(L), U0, exp_H, N, s_list, theta_list, k+1))

        E_list.append(E)
        V_list.append(V)


    return {
    "s": s_list,
    "theta": theta_list,
    "E": E_list,
    "V": V_list
    }


In [21]:
# Example usage of DB_QSP_outer
z_list = [-0.2, 0.1j, 0.01]
results = DB_QSP_outer(U0, H, exp_H_2, z_list, N=1)

Simulating 5 qubits.. |                                                      | [  0%]

Initial EV 1.4917303067381127 8.020949181291089                                      


In [22]:
print(results["E"])

[1.4917303067381127, 1.831675215391402, 2.0545245084385386, 2.1987443283650574]
